# 🏸 BMCA - Badminton Match Coaching Assistant

Upload a badminton match video and get AI-powered coaching insights.

**Steps:**
1. Run Setup cell (installs dependencies + downloads pipeline)
2. Upload your video
3. Run the pipeline
4. Download the report and load it in the local UI

In [ ]:
#@title 1. Setup (Run once) { display-mode: "form" }
!pip install -q torch torchvision ultralytics onnxruntime-gpu opencv-python-headless scipy pyyaml gdown tqdm lap

# Install mmcv — prefer pre-built wheel from Drive, fall back to source build (~5 min)
import subprocess, sys, os
try:
    import mmcv
    print(f'mmcv {mmcv.__version__} already installed')
except ImportError:
    # TODO: replace FILE_ID with your Google Drive file ID after uploading the wheel
    MMCV_DRIVE_FILE_ID = 'YOUR_FILE_ID_HERE'
    wheel_path = 'mmcv.whl'
    if MMCV_DRIVE_FILE_ID != 'YOUR_FILE_ID_HERE':
        import gdown
        print('Downloading pre-built mmcv wheel from Google Drive...')
        gdown.download(id=MMCV_DRIVE_FILE_ID, output=wheel_path, quiet=False)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', wheel_path])
        os.remove(wheel_path)
    else:
        print('No pre-built mmcv wheel — building from source (~5 min)...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
            '--no-binary', 'mmcv', 'mmcv==2.2.0'])

!pip install -q chumpy json-tricks matplotlib munkres xtcocotools pillow
!pip install -q --no-deps mmpose mmdet mmengine 2>/dev/null || true

import os
os.makedirs('bmca', exist_ok=True)

# Download pipeline script from GitHub
import urllib.request
url = 'https://raw.githubusercontent.com/sujkuttan/baddyCoach/master/colab/pipeline.py'
urllib.request.urlretrieve(url, 'bmca/pipeline.py')
print('✅ Setup complete! Pipeline downloaded.')

In [ ]:
#@title 2. Upload Video { display-mode: "form" }
import gdown

# Download test video from Google Drive
VIDEO_ID = '1aA_3keNIfCBjkNC9isovHwTQjfVejQdC'
video_path = '/content/test_match.mp4'
if not os.path.exists(video_path):
    print('Downloading test video...')
    gdown.download(id=VIDEO_ID, output=video_path, quiet=False)

print(f'\n✅ Video ready: {os.path.getsize(video_path) / 1024 / 1024:.1f} MB')
print(f'   Path: {video_path}')

In [ ]:
#@title 3. Run Pipeline { display-mode: "form" }
#@markdown On GPU (T4), expect ~5-15 min for a 30-min video.
POSE_MODEL = 'rtmpose' #@param ['rtmpose', 'mmpose']
#@markdown - `rtmpose`: Fast (default). RTMPose-M ONNX.
#@markdown - `mmpose`: Accurate. Auto-exports MMPose's default human model to ONNX on first run (~1 min).

!python -u /content/bmca/pipeline.py "{video_path}" --output /content/report.json --device cuda --pose-model {POSE_MODEL}

print('\n✅ Pipeline complete! Report: /content/report.json')

In [ ]:
#@title 4. View Summary & Download Report { display-mode: "form" }
import json, os
from google.colab import files

report_path = '/content/report.json'
if not os.path.exists(report_path):
    print('❌ Report not found. Run Step 3 first.')
else:
    report = json.load(open(report_path))

    print('═' * 50)
    print('  MATCH ANALYSIS SUMMARY')
    print('═' * 50)
    print(f'  Rallies: {len(report.get("rallies", []))}')
    print(f'  Shots: {report.get("shot_count", 0)}')
    print(f'  Players: {len(report.get("footwork", {}))}')

    sd = report.get('shot_distribution', {})
    if sd:
        print(f'\n  Top shots:')
        for s, p in sorted(sd.items(), key=lambda x: -x[1])[:5]:
            print(f'    {s:15s} {p*100:5.1f}%')

    if report.get('strengths'):
        print(f'\n  ✅ Strengths:')
        for s in report['strengths'][:3]: print(f'    • {s[:70]}')
    if report.get('weaknesses'):
        print(f'\n  ⚠️ Improve:')
        for w in report['weaknesses'][:3]: print(f'    • {w[:70]}')

    print('\n' + '═' * 50)
    files.download(report_path)
    print('📥 Downloaded! Load in local BMCA UI.')